# Gold: parent-child chunks

Task `gold_chunk` of the job `ingest_financebench`. Cuts the parser's elements in `silver_parsed` into **parents** (one
per section, given to the LLM) and **children** (small pieces, embedded and searched). A search finds children, and
the answer is written from their parents.

- **Parent:** a section. A new one starts at every `title` or `section_header` element, across the parts of a
  document. It holds all the section's elements: titles, headers, text, captions, footnotes, tables (HTML) and figure
  descriptions.
- **Child:** the section's text, cut by LangChain's `RecursiveCharacterTextSplitter` into pieces of about
  `CHUNK_SIZE` characters, plus the section's **tables**. A table of up to `TABLE_CHUNK_SIZE` characters is one child.
  A bigger one is cut **between rows** into groups, and every group repeats the table's header rows (if it has any), so each piece
  still says what its columns are. A table is never cut inside a row. Each child starts with its document and section
  header, such as `3M_2018_10K | Item 7. Management's Discussion`, so it can be read on its own.
- **Dropped:** page headers, footers and page numbers (noise). Every other element type the parser returns is kept.
  A figure is kept as its `description`, since its `content` is only a glyph.

**`gold_parents`**: one row per section.

| Column | Type | Holds |
| --- | --- | --- |
| `parent_id` | STRING | `doc_name#section_no`, such as `3M_2018_10K#42` |
| `doc_name` | STRING | The document name, such as `3M_2018_10K` |
| `section_no` | INT | The section's number in the document (0 = before the first header) |
| `header` | STRING | The section's title or section header (null for section 0) |
| `text` | STRING | All the section's elements (text, tables as HTML, figure descriptions, ...), joined by blank lines |
| `page_start` | INT | The section's first page in the PDF |
| `page_end` | INT | The section's last page in the PDF |

**`gold_children`**: one row per piece that is embedded and searched. Change data feed is on, so a Vector Search
index can sync from it.

| Column | Type | Holds |
| --- | --- | --- |
| `child_id` | STRING | `parent_id#kind#n`, such as `3M_2018_10K#42#text#0`, `3M_2018_10K#42#table#3-117` (a whole table) or `3M_2018_10K#42#table#3-117-2` (group 2 of a big table) |
| `parent_id` | STRING | The parent section it belongs to |
| `doc_name` | STRING | The document name |
| `kind` | STRING | `text` (a piece cut by the splitter) or `table` (a whole table, or a group of its rows) |
| `text` | STRING | `doc_name \| header`, a blank line, then the piece |
| `page_start` | INT | The first page: a text piece's parent's, or the table's own page |
| `page_end` | INT | The last page, the same way |

Pages are the real pages in the PDF.

**A stream on silver's change data feed** (`readChangeFeed`, `trigger(availableNow=True)`): a run reads the parts
that silver inserted or replaced since the last run (silver replaces a part when it repairs its parse), and stops. A section can run across parts, so a part is not chunked alone: every document with a new or replaced part is chunked
again from **all** its parts, and its rows in both tables are replaced (`replaceWhere`). The other documents are not
touched, and a rerun with nothing new does nothing. The steps below are functions because the stream calls them on
every micro-batch (`foreachBatch`).

Written to [doc/code_standard.md](../../doc/code_standard.md).

In [ ]:
%pip install langchain-text-splitters beautifulsoup4 -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql import Window, functions as F

In [ ]:
CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.finance_bench_silver"
GOLD_SCHEMA_NAME = "finance_bench_gold"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

PARSED_TABLE = f"{SILVER_SCHEMA}.silver_parsed"
PARENTS_TABLE = f"{GOLD_SCHEMA}.gold_parents"
CHILDREN_TABLE = f"{GOLD_SCHEMA}.gold_children"
CHECKPOINT = f"/Volumes/{CATALOG}/{GOLD_SCHEMA_NAME}/checkpoints/chunk_cdf"  # of the change data feed stream

CHUNK_SIZE = 2000  # characters, about 500 tokens
CHUNK_OVERLAP = 200
TABLE_CHUNK_SIZE = 4000  # characters of HTML, about 2,000 once the tags are removed for the embedding
HEADER_TYPES = ["title", "section_header"]  # each starts a new parent
DROPPED_TYPES = ["page_header", "page_footer", "page_number"]

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {GOLD_SCHEMA}.checkpoints")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PARENTS_TABLE} (
  parent_id STRING, doc_name STRING, section_no INT, header STRING, text STRING, page_start INT, page_end INT
)""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CHILDREN_TABLE} (
  child_id STRING, parent_id STRING, doc_name STRING, kind STRING, text STRING, page_start INT, page_end INT
) TBLPROPERTIES (delta.enableChangeDataFeed = true)""")  # change data feed: a Vector Search index can sync from it

## Step 1: the splitters, as pandas UDFs

Both run on the executors, a batch of values at a time.

**`split_text`**, LangChain's splitter, for the text of a section. It cuts at paragraphs first, then lines, then
words, so a piece stays as whole as `CHUNK_SIZE` allows.

**`split_table`**, for one table's HTML. LangChain has no splitter for this: its HTML splitters cut between headings
and either keep a table whole or flatten it to text and cut it in the middle of a row, so a piece of numbers loses
its column headers. `split_table` cuts only between rows (`<tr>`):

1. A table of up to `TABLE_CHUNK_SIZE` characters, or one with no rows, is returned whole, unchanged.
2. The **header** is the leading rows that have no `<td>` (only `<th>` cells). A table whose first row already has a
   `<td>` has no header, and its groups repeat nothing: such tables are lists (exhibits, a glossary, accounting
   standards), and their first row is data, not column names.
3. The other rows are packed into groups, in order, as many as fit in `TABLE_CHUNK_SIZE` with the header. A group
   always has at least one row, so a single huge row is not lost.
4. Each group becomes its own `<table>`: the header rows, then the group's rows.

**Example.** One value in, an array of pieces out:

| input | output |
| --- | --- |
| `split_text`: 600 characters, such as section 3's text without its table | `["(Mark One)⏎⏎A black square icon…⏎⏎TRANSITION REPORT…"]`: shorter than 2,000, so one piece |
| `split_text`: 5,000 characters in 10 paragraphs | `[piece 0 (~1,900), piece 1 (~1,950), piece 2 (~1,550)]`: each cut at a paragraph break, and each piece starts with up to 200 characters from the end of the one before |
| `split_table`: section 3's table, `<table><tr><th>Commission File Number</th>…</table>` (900 characters) | `["<table><tr><th>Commission File Number</th>…</table>"]`: up to 4,000, so the same table, whole |
| `split_table`: an exhibit list of 6,500 characters, 30 rows of `<td>` and no `<th>`, such as `<tr><td>3.1</td><td>Sixth Restated Certificate of Incorporation…</td></tr>` | `["<table><tr><td>3.1</td>…</tr>…</table>", "<table><tr><td>10.8</td>…</tr>…</table>"]`: 2 groups of about 15 rows, with no header row repeated |
| `split_table`: a balance sheet of 9,800 characters, 1 `<th>` row (`(in millions)`, `2020`, `2019`) and 60 rows of `<td>` | `["<table><tr><th>(in millions)</th>…</tr><tr><td>Cash</td>…</tr>…</table>", "<table><tr><th>(in millions)</th>…</tr><tr><td>Accounts receivable</td>…</table>", "<table><tr><th>(in millions)</th>…</tr>…<tr><td>Total liabilities and equity</td>…</table>"]`: 3 groups of about 20 rows, each under 4,000 characters and each starting with the header row |

In [ ]:
@F.pandas_udf("array<string>")
def split_text(texts: pd.Series) -> pd.Series:
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    return texts.apply(splitter.split_text)


@F.pandas_udf("array<string>")
def split_table(tables: pd.Series) -> pd.Series:
    def groups(html):
        rows = BeautifulSoup(html, "html.parser").find_all("tr")
        if len(html) <= TABLE_CHUNK_SIZE or not rows:
            return [html]
        n_header = next((i for i, row in enumerate(rows) if row.find("td")), len(rows))  # leading <th>-only rows
        header = "".join(map(str, rows[:n_header]))
        pieces = [""]
        for row in map(str, rows[n_header:]):
            if pieces[-1] and len(header) + len(pieces[-1]) + len(row) > TABLE_CHUNK_SIZE:
                pieces.append("")
            pieces[-1] += row
        return [f"<table>{header}{piece}</table>" for piece in pieces]

    return tables.apply(groups)

## Step 2: one row per element, with its section

For the documents of the micro-batch, reads all their parts in `silver_parsed` and explodes the parser's `elements` in reading order, with their type, content and real page, and drops the
`DROPPED_TYPES`. A figure takes its `description` (its `content` is only a glyph). `section_no` is a running count of the headers in the document, so every element gets the
section of the last header before it. Elements before the first header are section 0.

**Example.** Part 1 of `PGE_2020_10K` (`page_offset` 0), the first page of the filing. The parser returned 9
elements. Long text is cut with `…`, and `⏎` is a line break.

**2a. After `posexplode`.** One row per element. `pos` is its place in the part's reading order, and `e` is the
element, still a VARIANT:

| doc_name | part_no | page_offset | pos | e |
| --- | --- | --- | --- | --- |
| PGE_2020_10K | 1 | 0 | 0 | `{"type": "title", "content": "UNITED STATES⏎SECURITIES AND EXCHANGE COMMISSION", "bbox": [{"page_id": 0, …}], …}` |
| PGE_2020_10K | 1 | 0 | 1 | `{"type": "text", "content": "Washington, D.C. 20549", …}` |
| PGE_2020_10K | 1 | 0 | 2 | `{"type": "section_header", "content": "FORM 10-K", …}` |
| PGE_2020_10K | 1 | 0 | 3 | `{"type": "section_header", "content": "(Mark One)", …}` |
| PGE_2020_10K | 1 | 0 | 4 | `{"type": "figure", "content": "☒", "description": "A black square icon containing a diagonal cross…", …}` |
| PGE_2020_10K | 1 | 0 | 5 | `{"type": "text", "content": "ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d)…", …}` |
| PGE_2020_10K | 1 | 0 | 6 | `{"type": "figure", "content": "☐", "description": "A simple black square outline…", …}` |
| PGE_2020_10K | 1 | 0 | 7 | `{"type": "text", "content": "TRANSITION REPORT PURSUANT TO SECTION 13…", …}` |
| PGE_2020_10K | 1 | 0 | 8 | `{"type": "table", "content": "<table><tr><th>Commission File Number</th>…", …}` |

**2b. After the second `select`.** The VARIANT becomes columns. A figure takes its `description`, and
`page` = `page_offset` + `bbox[0].page_id` + 1:

| doc_name | part_no | pos | type | content | page |
| --- | --- | --- | --- | --- | --- |
| PGE_2020_10K | 1 | 0 | title | UNITED STATES⏎SECURITIES AND EXCHANGE COMMISSION | 1 |
| PGE_2020_10K | 1 | 1 | text | Washington, D.C. 20549 | 1 |
| PGE_2020_10K | 1 | 2 | section_header | FORM 10-K | 1 |
| PGE_2020_10K | 1 | 3 | section_header | (Mark One) | 1 |
| PGE_2020_10K | 1 | 4 | figure | A black square icon containing a diagonal cross… | 1 |
| PGE_2020_10K | 1 | 5 | text | ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d)…⏎For the Fiscal Year Ended December 31, 2020 | 1 |
| PGE_2020_10K | 1 | 6 | figure | A simple black square outline… | 1 |
| PGE_2020_10K | 1 | 7 | text | TRANSITION REPORT PURSUANT TO SECTION 13…⏎For the transition period from… | 1 |
| PGE_2020_10K | 1 | 8 | table | `<table><tr><th>Commission File Number</th>…<td>PG&E CORPORATION</td>…</table>` | 1 |

**2c. After `filter`.** Removes the `DROPPED_TYPES` (`page_header`, `page_footer`, `page_number`) and rows with
empty content. This page has none, so all 9 rows stay. A page number "1" at the foot of the page would be removed
here.

**2d. After `withColumn("section_no")`.** A running count of headers (`title` or `section_header`), in the order
`part_no`, `pos` over the whole document:

| pos | type | is a header | section_no |
| --- | --- | --- | --- |
| 0 | title | 1 | **1** |
| 1 | text | 0 | 1 |
| 2 | section_header | 1 | **2** |
| 3 | section_header | 1 | **3** |
| 4 | figure | 0 | 3 |
| 5 | text | 0 | 3 |
| 6 | figure | 0 | 3 |
| 7 | text | 0 | 3 |
| 8 | table | 0 | 3 |

The count goes on into part 2, so a section that starts in part 1 and ends in part 2 stays one section.

In [ ]:
def elements_of(session, docs):
    return (
        session.table(PARSED_TABLE).filter(F.col("doc_name").isin(docs))
        .select("doc_name", "part_no", "page_offset",
                F.posexplode(F.expr("cast(parsed:document:elements AS array<variant>)")).alias("pos", "e"))
        .select(
            "doc_name", "part_no", "pos",
            F.expr("variant_get(e, '$.type', 'string')").alias("type"),
            F.expr("CASE WHEN variant_get(e, '$.type', 'string') = 'figure' THEN variant_get(e, '$.description', 'string')"
                   " ELSE variant_get(e, '$.content', 'string') END").alias("content"),
            (F.col("page_offset") + F.expr("variant_get(e, '$.bbox[0].page_id', 'int')") + 1).alias("page"),
        )
        .filter(~F.col("type").isin(DROPPED_TYPES) & (F.trim("content") != ""))
        .withColumn("section_no", F.sum(F.col("type").isin(HEADER_TYPES).cast("int"))
                    .over(Window.partitionBy("doc_name").orderBy("part_no", "pos")).cast("int"))
    )

## Step 3: the parents

All the elements of a section, in reading order, joined by blank lines. The header is the section's first element
when it is a title or section header (section 0 has none).

**3a. After `groupBy` and `agg`.** One row per section. `els` holds its elements, sorted by `part_no` and `pos`:

| doc_name | section_no | els |
| --- | --- | --- |
| PGE_2020_10K | 1 | `[{1, 0, title, "UNITED STATES⏎SECURITIES…", 1}, {1, 1, text, "Washington, D.C. 20549", 1}]` |
| PGE_2020_10K | 2 | `[{1, 2, section_header, "FORM 10-K", 1}]` |
| PGE_2020_10K | 3 | `[{1, 3, section_header, "(Mark One)", 1}, {1, 4, figure, "A black square icon…", 1}, …, {1, 8, table, "<table>…", 1}]` |

**3b. After `select`.** These are the rows Step 5 writes to `gold_parents`:

| parent_id | doc_name | section_no | header | text | page_start | page_end |
| --- | --- | --- | --- | --- | --- | --- |
| PGE_2020_10K#1 | PGE_2020_10K | 1 | UNITED STATES⏎SECURITIES AND EXCHANGE COMMISSION | UNITED STATES⏎SECURITIES AND EXCHANGE COMMISSION⏎⏎Washington, D.C. 20549 | 1 | 1 |
| PGE_2020_10K#2 | PGE_2020_10K | 2 | FORM 10-K | FORM 10-K | 1 | 1 |
| PGE_2020_10K#3 | PGE_2020_10K | 3 | (Mark One) | (Mark One)⏎⏎A black square icon…⏎⏎ANNUAL REPORT…⏎⏎A simple black square…⏎⏎TRANSITION REPORT…⏎⏎`<table>…</table>` | 1 | 1 |

On a real filing, a section such as "Item 7. Management's Discussion and Analysis" holds many pages, so
`page_start` and `page_end` differ.

In [ ]:
def parents_of(elements):
    return (
        elements.groupBy("doc_name", "section_no")
        .agg(F.sort_array(F.collect_list(F.struct("part_no", "pos", "type", "content", "page"))).alias("els"))
        .select(
            F.concat_ws("#", "doc_name", F.col("section_no").cast("string")).alias("parent_id"),
            "doc_name", "section_no",
            F.when(F.col("els")[0]["type"].isin(HEADER_TYPES), F.col("els")[0]["content"]).alias("header"),
            F.array_join("els.content", "\n\n").alias("text"),
            F.array_min("els.page").alias("page_start"),
            F.array_max("els.page").alias("page_end"),
        )
    )

## Step 4: the children

The text children are each section's non-table elements, joined and cut by the splitter. They take the pages of their
parent. The table children are each table cut by `split_table` (Step 1): one child for a table that fits, one per
group of rows for a bigger one, all on the table's own page. Both start with the document and the section header.

**4a. `text_children`.** Without the table, joined, then `posexplode` of the splitter's pieces (`n` = the piece's
number). Sections 1 and 2 are short, so each gives one piece:

| doc_name | section_no | kind | n | text | page |
| --- | --- | --- | --- | --- | --- |
| PGE_2020_10K | 1 | text | 0 | UNITED STATES⏎SECURITIES AND EXCHANGE COMMISSION⏎⏎Washington, D.C. 20549 | null |
| PGE_2020_10K | 2 | text | 0 | FORM 10-K | null |
| PGE_2020_10K | 3 | text | 0 | (Mark One)⏎⏎A black square icon…⏎⏎ANNUAL REPORT…⏎⏎A simple black square…⏎⏎TRANSITION REPORT… | null |

**4b. `table_children`.** `split_table` on each table, then `posexplode` of its pieces (`g` = the group's number).
A table that stays whole keeps `n` = `part_no-pos`, so its `child_id` (and its vector in `gold_embeddings`) stays the
same as before tables were cut. A table cut into groups gets `n` = `part_no-pos-g`. Section 3's table is 900
characters, so it is one piece:

| doc_name | section_no | kind | n | text | page |
| --- | --- | --- | --- | --- | --- |
| PGE_2020_10K | 3 | table | 1-8 | `<table><tr><th>Commission File Number</th>…</table>` | 1 |

A big table gives one row per group. Such as the balance sheet of Step 1, in a later section of the same filing
(section 88, part 3, element 45, page 112):

| doc_name | section_no | kind | n | text | page |
| --- | --- | --- | --- | --- | --- |
| PGE_2020_10K | 88 | table | 3-45-0 | `<table><tr><th>(in millions)</th><th>2020</th><th>2019</th></tr><tr><td>Cash</td>…</table>` | 112 |
| PGE_2020_10K | 88 | table | 3-45-1 | `<table><tr><th>(in millions)</th><th>2020</th><th>2019</th></tr><tr><td>Accounts receivable</td>…</table>` | 112 |
| PGE_2020_10K | 88 | table | 3-45-2 | `<table><tr><th>(in millions)</th><th>2020</th><th>2019</th></tr>…<tr><td>Total liabilities and equity</td>…</table>` | 112 |

**4c. After `unionByName`, `join` with the parents and `select`.** These are the rows Step 5 writes to `gold_children`. The join brings
`parent_id`, `header` and the parent's pages. `text` gets the `doc_name | header` prefix, and a text piece takes its
parent's pages:

| child_id | parent_id | doc_name | kind | text | page_start | page_end |
| --- | --- | --- | --- | --- | --- | --- |
| PGE_2020_10K#1#text#0 | PGE_2020_10K#1 | PGE_2020_10K | text | PGE_2020_10K \| UNITED STATES⏎SECURITIES…⏎⏎UNITED STATES⏎SECURITIES…⏎⏎Washington, D.C. 20549 | 1 | 1 |
| PGE_2020_10K#2#text#0 | PGE_2020_10K#2 | PGE_2020_10K | text | PGE_2020_10K \| FORM 10-K⏎⏎FORM 10-K | 1 | 1 |
| PGE_2020_10K#3#text#0 | PGE_2020_10K#3 | PGE_2020_10K | text | PGE_2020_10K \| (Mark One)⏎⏎(Mark One)⏎⏎A black square icon…⏎⏎ANNUAL REPORT… | 1 | 1 |
| PGE_2020_10K#3#table#1-8 | PGE_2020_10K#3 | PGE_2020_10K | table | PGE_2020_10K \| (Mark One)⏎⏎`<table><tr><th>Commission File Number</th>…</table>` | 1 | 1 |
| … | | | | | | |
| PGE_2020_10K#88#table#3-45-0 | PGE_2020_10K#88 | PGE_2020_10K | table | PGE_2020_10K \| Consolidated Balance Sheets⏎⏎`<table><tr><th>(in millions)</th>…<td>Cash</td>…</table>` | 112 | 112 |
| PGE_2020_10K#88#table#3-45-1 | PGE_2020_10K#88 | PGE_2020_10K | table | PGE_2020_10K \| Consolidated Balance Sheets⏎⏎`<table><tr><th>(in millions)</th>…<td>Accounts receivable</td>…</table>` | 112 | 112 |
| PGE_2020_10K#88#table#3-45-2 | PGE_2020_10K#88 | PGE_2020_10K | table | PGE_2020_10K \| Consolidated Balance Sheets⏎⏎`<table><tr><th>(in millions)</th>…<td>Total liabilities and equity</td>…</table>` | 112 | 112 |

A search that finds `PGE_2020_10K#3#table#1-8` (such as "PG&E's IRS employer identification number") returns its
parent `PGE_2020_10K#3`, the whole section, to the LLM. A search for "PG&E's total liabilities in 2020" finds
`PGE_2020_10K#88#table#3-45-2`, the group with that row and the column headers, and the LLM reads its parent
`PGE_2020_10K#88`, with the whole balance sheet.

In [ ]:
def children_of(elements, parents):
    text_children = (
        elements.filter(F.col("type") != "table")
        .groupBy("doc_name", "section_no")
        .agg(F.sort_array(F.collect_list(F.struct("part_no", "pos", "content"))).alias("els"))
        .select("doc_name", "section_no", F.lit("text").alias("kind"),
                F.posexplode(split_text(F.array_join("els.content", "\n\n"))).alias("n", "text"),
                F.lit(None).cast("int").alias("page"))
    )
    table_children = (
        elements.filter(F.col("type") == "table")
        .withColumn("pieces", split_table("content"))
        .select("doc_name", "section_no", "part_no", "pos", "page", F.size("pieces").alias("n_pieces"),
                F.posexplode("pieces").alias("g", "text"))
        .select("doc_name", "section_no", F.lit("table").alias("kind"),
                F.when(F.col("n_pieces") == 1, F.concat_ws("-", "part_no", "pos"))  # a whole table keeps its old id
                .otherwise(F.concat_ws("-", "part_no", "pos", "g")).alias("n"), "text", "page")
    )
    return (
        text_children.withColumn("n", F.col("n").cast("string")).unionByName(table_children)
        .join(parents.select("doc_name", "section_no", "parent_id", "header", "page_start", "page_end"),
              ["doc_name", "section_no"])
        .select(
            F.concat_ws("#", "parent_id", "kind", "n").alias("child_id"),
            "parent_id", "doc_name", "kind",
            F.concat(F.concat_ws(" | ", "doc_name", "header"), F.lit("\n\n"), "text").alias("text"),
            F.coalesce("page", "page_start").alias("page_start"),
            F.coalesce("page", "page_end").alias("page_end"),
        )
    )

## Step 5: the stream

Reads silver's change data feed and keeps the rows that are a part's new content: `insert` (a new part) and
`update_postimage` (a part whose parse silver repaired). `update_preimage`, the old content, is dropped. On its first
run, with no checkpoint, the stream reads the whole table as inserts, so every document is chunked once. For every
micro-batch, `chunk_docs` takes the documents in it, chunks them
again from all their parts (Steps 2 to 4) and replaces their rows in both tables. `replaceWhere` makes each write
one commit, which deletes the document's old rows and adds the new ones, so a document whose sections changed keeps
no stale rows. Only the document names are collected to the driver, not the data.

**Example.** An earlier run chunked `3M_2018_10K` and part 2 of `PGE_2020_10K`, which had lost pages to the
parser's rate limit. Since then silver has inserted part 1 of `AMD_2022_10K` and repaired part 2 of `PGE_2020_10K`.

**5a. After `readStream` with `readChangeFeed`.** The change rows since the last run:

| doc_name | path | part_no | page_offset | parsed | parsed_at | _change_type |
| --- | --- | --- | --- | --- | --- | --- |
| PGE_2020_10K | …/pdfs/PGE_2020_10K.pdf | 2 | 50 | `{"document": {"elements": […]}, "error_status": [{"error_message": "Request rate limit exceeded…"}, …]}` | 2026-11-20 09:31:47 | update_preimage |
| PGE_2020_10K | …/pdfs/PGE_2020_10K.pdf | 2 | 50 | `{"document": {"elements": […]}, …}` | 2026-11-27 10:02:41 | update_postimage |
| AMD_2022_10K | …/pdfs/AMD_2022_10K.pdf | 1 | 0 | `{"document": {"elements": […]}, …}` | 2026-11-27 10:02:41 | insert |

**5b. After `filter`.** The old content of `PGE_2020_10K` part 2 is dropped:

| doc_name | part_no | parsed_at | _change_type |
| --- | --- | --- | --- |
| PGE_2020_10K | 2 | 2026-11-27 10:02:41 | update_postimage |
| AMD_2022_10K | 1 | 2026-11-27 10:02:41 | insert |

**5c. `docs` and `only_docs`.** `["PGE_2020_10K", "AMD_2022_10K"]` and `doc_name IN ('PGE_2020_10K', 'AMD_2022_10K')`.

**5d. After the writes.** In `gold_parents` (the same in `gold_children`), only the rows of the two documents are
replaced:

| doc_name | rows before | rows after |
| --- | --- | --- |
| 3M_2018_10K | 812 | 812, not touched |
| PGE_2020_10K | its sections, part 2 with pages missing | its sections from parts 1 and 2, nothing missing |
| AMD_2022_10K | 0 | its sections |

`PGE_2020_10K` is chunked again from all its parts, so a section that runs from part 1 into the repaired part 2 is
whole again. A rerun with nothing new runs no micro-batch.

In [ ]:
def chunk_docs(batch, batch_id):
    docs = [row.doc_name for row in batch.select("doc_name").distinct().collect()]  # a few names, not the data
    if not docs:  # a batch of only old content (update_preimage) changes no document
        return
    only_docs = "doc_name IN (" + ", ".join(f"'{doc}'" for doc in docs) + ")"
    elements = elements_of(batch.sparkSession, docs)  # inside foreachBatch, the batch's session, not `spark`
    parents = parents_of(elements)
    parents.write.mode("overwrite").option("replaceWhere", only_docs).saveAsTable(PARENTS_TABLE)
    children_of(elements, parents).write.mode("overwrite").option("replaceWhere", only_docs).saveAsTable(CHILDREN_TABLE)


(
    spark.readStream.option("readChangeFeed", "true").table(PARSED_TABLE)
    .filter(F.col("_change_type").isin("insert", "update_postimage"))  # a part's new content, not the old
    .writeStream.foreachBatch(chunk_docs)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

In [ ]:
display(spark.sql(f"""
SELECT (SELECT count(DISTINCT doc_name) FROM {PARENTS_TABLE}) AS documents,
       (SELECT count(*) FROM {PARENTS_TABLE}) AS parents,
       (SELECT count_if(kind = 'text') FROM {CHILDREN_TABLE}) AS text_children,
       (SELECT count_if(kind = 'table') FROM {CHILDREN_TABLE}) AS table_children,
       (SELECT max(length(text)) FROM {CHILDREN_TABLE} WHERE kind = 'text') AS longest_text_child
"""))